---
# 7. Data Sources

Product settlements are driven by real-world data observed during the session window (12pm to 12pm London).
Below are helpers to fetch each data source.

## 7a. Weather — Open-Meteo (free, no API key)

15-minute resolution weather observations and forecasts for London via [Open-Meteo](https://open-meteo.com/).
Returns temperature (°C), wind speed, humidity (%), precipitation, cloud cover, visibility, and apparent temperature.

**Relevant for:** WX_SPOT (temp_F × humidity at 12pm), WX_SUM (15-min aggregate / 100).

In [ ]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """15-min weather for London. 96 steps = 24 hours.

    Returns DataFrame with: time, temperature, wind_speed, humidity,
    precipitation, cloud_cover, visibility, apparent_temperature.
    """
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "apparent_temperature": m["apparent_temperature"],
        "humidity": m["relative_humidity_2m"],
        "precipitation": m["precipitation"],
        "wind_speed": m["wind_speed_10m"],
        "cloud_cover": m["cloud_cover"],
        "visibility": m["visibility"],
    })

In [ ]:
df_weather = get_weather()
print(f"{len(df_weather)} readings, {df_weather.time.min()} -> {df_weather.time.max()}")
df_weather.tail(5)

## 7b. Thames Tidal Level — EA Flood Monitoring API (free, no API key)

Tidal level readings at the Westminster gauge, sampled every 15 minutes.
Levels are in **metres Above Ordnance Datum (mAOD)** — the UK height reference where 0 mAOD = mean sea level at Newlyn, Cornwall. Negative values mean the water surface is below the datum.

**Relevant for:** TIDE_SPOT (abs value in mm at 12pm), TIDE_SWING (strangle on 15-min diffs in cm).

In [ ]:
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

def get_thames(limit=200):
    """Fetch recent Thames tidal readings at Westminster.

    Returns DataFrame with: time, level (mAOD).
    Use limit=400 for ~4 days of history.
    """
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": limit},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])
    df = pd.DataFrame(items)[["dateTime", "value"]].rename(columns={"dateTime": "time", "value": "level"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/London")
    return df.sort_values("time").reset_index(drop=True)

In [ ]:
df_thames = get_thames(limit=200)
print(f"{len(df_thames)} readings, {df_thames.time.min()} -> {df_thames.time.max()}")
df_thames.tail(5)

## 7c. Flights — AeroDataBox via RapidAPI

Arrivals and departures at London Heathrow via the [AeroDataBox](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/) flights endpoint. You'll need a free RapidAPI key — sign up [here](https://rapidapi.com/auth/sign-up) and subscribe to the Basic (free) tier. The free plan is limited to ~150 requests/month so be resourceful. Feel free to look for alternative endpoints, but settlement will be based on this data provider.

Two query styles are available (max 12h window each):
- **By relative time:** `offset_minutes` + `duration_minutes` relative to now
- **By time range:** explicit local times `fromLocal` / `toLocal` (format: `YYYY-MM-DDTHH:mm`)

The API also supports several boolean filters — check the [AeroDataBox docs](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/playground/apiendpoint_3dbf8f9a-22de-4a99-8e7d-e542f6e63e4f) to learn what's available.

**Relevant for:** LHR_COUNT (total flights in 24h), LHR_INDEX (imbalance metric per 30-min interval).

In [2]:
import requests
import json
import os
import pandas as pd
from datetime import datetime
import time

AERODATABOX_KEY = "954aeb65f0mshc43dfde89d2698bp14c7ddjsnc2bec6059409"  # Replace with your key
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

def fetch_combined_flights(offsets=[-720, 0, 720], duration=720):
    """
    Fetches flight data for multiple time windows and combines them.
    """
    master_data = {"arrivals": [], "departures": []}
    
    for offset in offsets:
        params = f"?offsetMinutes={offset}&durationMinutes={duration}&direction=Both"
        url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{AIRPORT}{params}"
        
        print(f"Fetching window: offset {offset}...")
        resp = requests.get(url, headers={
            "x-rapidapi-host": AERODATABOX_HOST, 
            "x-rapidapi-key": AERODATABOX_KEY
        })
        
        if resp.status_code == 200:
            data = resp.json()
            master_data["arrivals"].extend(data.get('arrivals', []))
            master_data["departures"].extend(data.get('departures', []))
        else:
            print(f"Failed to fetch window {offset}: {resp.status_code}")
        
        # Respect API rate limits (1 request per second)
        time.sleep(1.1)

    # Save the combined JSON
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    json_filename = f"flights_LHR_combined_{timestamp}.json"
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(master_data, f, indent=4)
    
    print(f"Combined data saved to {json_filename}")
    return master_data

def process_and_save_flights(data):
    """Filters for unique operators and saves to CSV."""
    
    def flatten_flights(flight_list):
        rows = []
        seen_ids = set() # Prevent duplicates across overlapping windows
        for f in flight_list:
            f_id = f.get('number')
            if f.get('codeshareStatus') == 'IsOperator' and f_id not in seen_ids:
                rows.append({
                    'flight_number': f_id,
                    'airline': f.get('airline', {}).get('name'),
                    'status': f.get('status'),
                    'scheduled_local': f.get('movement', {}).get('scheduledTime', {}).get('local')
                })
                seen_ids.add(f_id)
        return pd.DataFrame(rows)

    df_arrivals = flatten_flights(data.get('arrivals', []))
    df_departures = flatten_flights(data.get('departures', []))

    df_arrivals.to_csv('lhr_arrivals_combined.csv', index=False)
    df_departures.to_csv('lhr_departures_combined.csv', index=False)

    print(f"Saved {len(df_arrivals)} arrivals and {len(df_departures)} departures to CSV.")

# Run the process
combined_data = fetch_combined_flights()
process_and_save_flights(combined_data)

Fetching window: offset -720...
Fetching window: offset 0...
Fetching window: offset 720...
Combined data saved to flights_LHR_combined_20260228_182848.json
Saved 678 arrivals and 707 departures to CSV.


In [ ]:
# Fetch flights from the last 12 hours
data = fetch_flights(offset_minutes=-720, duration_minutes=720)
print(f"{len(data.get('arrivals', []))} arrivals, {len(data.get('departures', []))} departures")

# Look at one flight record to understand the structure
if data.get("arrivals"):
    print("\nExample arrival record:")
    print(json.dumps(data["arrivals"][0], indent=2))

---
# 8. Price-Time Priority

The exchange uses **price-time priority** for order matching:

1. **Price priority** — an incoming buy order fills against the **lowest-priced** sell order first. An incoming sell order fills against the **highest-priced** buy order first.
2. **Time priority** — at the same price level, the order that was placed **first** gets filled first.

This means if you and another participant both quote at the same price, whoever was there first gets the fill. If you cancel and re-place your order, you go to the **back of the queue** at that price level.

---
# 9. Simple Quoter

A minimal market-making loop: every N seconds, cancel all orders, compute mid, place a bid and ask at fixed width around mid.

In [ ]:
import math


class SimpleQuoter(BaseBot):
    """Cancel-and-replace quoter. Quotes fixed width around mid on all products."""

    def on_orderbook(self, ob):
        pass

    def on_trades(self, trade: Trade):
        side = "BOUGHT" if trade.buyer == self.username else "SOLD"
        print(f"  FILL: {side} {trade.volume}x {trade.product} @ {trade.price}")

    def run_loop(self, width=5.0, volume=5, interval=5):
        interval = max(interval, 1)
        products = {p.symbol: p for p in self.get_products()}
        self.start()  # start SSE so on_trades fires

        while True:
            self.cancel_all_orders()

            for symbol, product in products.items():
                ob = self.get_orderbook(symbol)

                # Mid = (best_bid + best_ask) / 2, ignoring our own orders
                bids = [o.price for o in ob.buy_orders if o.volume - o.own_volume > 0]
                asks = [o.price for o in ob.sell_orders if o.volume - o.own_volume > 0]
                mid = (max(bids) + min(asks)) / 2 if bids and asks else product.startingPrice

                tick = product.tickSize
                bid = math.floor((mid - width) / tick) * tick
                ask = math.ceil((mid + width) / tick) * tick

                if bid > 0 and bid < ask:
                    self.send_orders([
                        OrderRequest(symbol, bid, Side.BUY, volume),
                        OrderRequest(symbol, ask, Side.SELL, volume),
                    ])
                    print(f"  {symbol}  mid={mid:.0f}  {volume}@{bid} / {volume}@{ask}")

            time.sleep(interval)

In [ ]:
# Run the quoter — interrupt the cell to stop

quoter = SimpleQuoter(EXCHANGE_URL, USERNAME, PASSWORD)
print("Starting quoter (Ctrl+C to stop)...")
try:
    quoter.run_loop(width=5, volume=5, interval=5)
except KeyboardInterrupt:
    quoter.cancel_all_orders()
    quoter.stop()
    print("Quoter stopped.")

---
# Next Steps

This quoter is intentionally simple and missing functionality you might want to add. Here are a few ideas to improve it:

- **Use data sources** — adjust fair value based on weather, flights, or tides instead of only following mid (with a fallback to the starting price)
- **Adjust width** based on your confidence in your theo (your theoretical valuation)
- **Manage risk** — stop quoting one side when your position gets too large
- **Multiple levels** — quote at several price levels to provide more volume
- **Cross-product** — if products are related, use one to help price the other
- **Be smarter about cancels** — only requote when mid actually changes to preserve queue priority